In [3]:
import pandas as pd
import numpy as np
import time
import os
from numba import njit, prange

# Charger le fichier CSV
df = pd.read_csv("notes_etudiants.csv")

# Colonnes en NumPy
maths = df["Maths"].values
physique = df["Physique"].values
anglais = df["Anglais"].values

# --- Version séquentielle ---
def moyenne_seq(m, p, a):
    return (m*5 + p*4 + a*2) / 11

start = time.time()
moy_seq = [moyenne_seq(m, p, a) for m, p, a in zip(maths, physique, anglais)]
t_seq = time.time() - start
print("Temps séquentiel :", t_seq, "secondes")

# --- Version parallèle ---
@njit(parallel=True)
def moyenne_parallel(maths, physique, anglais):
    N = len(maths)
    moyennes = np.empty(N)
    for i in prange(N):
        moyennes[i] = (maths[i]*5 + physique[i]*4 + anglais[i]*2) / 11
    return moyennes

# Compilation JIT
_ = moyenne_parallel(maths, physique, anglais)

# Mesure du temps parallèle
start = time.time()
moy_parallel = moyenne_parallel(maths, physique, anglais)
t_par = time.time() - start
print("Temps parallèle :", t_par, "secondes")

# --- Calcul du speedup ---
speedup = t_seq / t_par
print("Speedup =", speedup)

# Nombre de cœurs disponibles
N = os.cpu_count()

# Calcul de la proportion parallélisable (bornée à 100%)
P = min((1 - 1/speedup) / (1 - 1/N), 1.0)

print("Nombre de cœurs utilisés :", N)
print("Proportion parallélisable (P) =", P*100, "%")


Temps séquentiel : 2.187368869781494 secondes
Temps parallèle : 0.005225181579589844 secondes
Speedup = 418.62064245300235
Nombre de cœurs utilisés : 8
Proportion parallélisable (P) = 100.0 %
